# Applying Technical Indicators
Produce 2 parquets:
1. Time series data and technical indicators 
2. Time series data and technical indicators with NLP features

2 were produced to see the effect of dropping NAN rows in future ablations, is NLP worth the dropped columns?

In [1]:
import os
import site

import os
import warnings
import pandas as pd
import numpy as np
import pandas_ta as ta

warnings.filterwarnings('ignore')

/opt/anaconda3/envs/ta_env/lib/python3.12/site-packages/pandas_ta/__init__.py:7: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


## Applying Technical Indicators
Time series data and Technical Indicators Parquet Produced

In [2]:
def apply_ta_indicators(df_group):
    df_group.set_index(pd.DatetimeIndex(df_group['date']), inplace=True)
    #Trend
    df_group.ta.ema(length=12, append=True)
    df_group.ta.ema(length=26, append=True)
    df_group.ta.ema(length=50, append=True)

    df_group.ta.macd(fast=12, slow=26, signal=9, append=True)



    df_group.ta.rsi(length=14, append=True)
    df_group.ta.stochrsi(length=14, append=True)


    df_group.ta.atr(length=14, append=True)

    bb = ta.bbands(df_group['close'], length=20, std=2)
    df_group['BB_upper'] = bb['BBU_20_2.0']
    df_group['BB_middle'] = bb['BBM_20_2.0']
    df_group['BB_lower'] = bb['BBL_20_2.0']


    df_group.ta.obv(append=True)
    return df_group.reset_index(drop=True)

In [3]:
companies = pd.read_parquet('../data/dataset/stock_table.parquet')
stocks = pd.read_parquet('../data/dataset/stock_prices.parquet')

companies.columns = [x.lower() for x in companies.columns]
stocks.columns = [x.lower() for x in stocks.columns]

if 'symbol' in companies.columns:
    companies = companies.rename(columns={'symbol': 'ticker'})

if 'ticker' not in companies.columns:
    raise KeyError("'ticker' not found in companies columns")

stocks['date'] = pd.to_datetime(stocks['date'])

print(f"Shape of stocks raw: {stocks.shape}")

stocks = stocks.sort_values(by=['ticker', 'date'])

stocks_ta = stocks.groupby('ticker', group_keys=False).apply(apply_ta_indicators)

print(f"Shape after TA indicators: {stocks_ta.shape}")


Shape of stocks raw: (108592, 8)
Shape after TA indicators: (108592, 22)


In [4]:
stocks_ta

,date,open,high,low,close,adj close,volume,ticker,EMA_12,EMA_26,...,MACDh_12_26_9,MACDs_12_26_9,RSI_14,STOCHRSIk_14_14_3_3,STOCHRSId_14_14_3_3,ATRr_14,BB_upper,BB_middle,BB_lower,OBV
0,2012-09-04,95.108574,96.448570,94.928574,96.424286,87.121140,91973000.0,AAPL,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,91973000.0
1,2012-09-05,96.510002,96.621429,95.657143,95.747147,86.509338,84093800.0,AAPL,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7879200.0
2,2012-09-06,96.167145,96.898575,95.828575,96.610001,87.288956,97799100.0,AAPL,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,105678300.0
3,2012-09-07,96.864288,97.497147,96.538574,97.205711,87.827171,82416600.0,AAPL,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,188094900.0
4,2012-09-10,97.207146,97.612854,94.585716,94.677139,85.542564,121999500.0,AAPL,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,66095400.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1253,2017-08-28,76.900002,76.940002,76.260002,76.470001,76.470001,8229700.0,XOM,77.187452,78.267858,...,-0.107548,-0.972858,31.975492,35.117121,31.775404,0.786087,81.525829,78.2435,74.961171,-268825100.0
1254,2017-08-29,76.209999,76.489998,76.080002,76.449997,76.449997,7060400.0,XOM,77.073998,78.133202,...,-0.069077,-0.990127,31.851847,48.597552,38.712818,0.759224,81.303475,78.0575,74.811525,-275885500.0
1255,2017-08-30,76.239998,76.449997,76.059998,76.099998,76.099998,8218000.0,XOM,76.924151,77.982594,...,-0.054652,-1.003790,29.688704,55.025431,46.246701,0.732850,80.964170,77.8325,74.700830,-284103500.0
1256,2017-08-31,76.269997,76.489998,76.050003,76.330002,76.330002,15641700.0,XOM,76.832744,77.860180,...,-0.018917,-1.008519,32.913052,73.940933,59.187972,0.711932,80.569554,77.6245,74.679446,-268461800.0


In [5]:
columns_to_check = ['EMA_12', 'EMA_26','EMA_50','MACD_12_26_9','MACDh_12_26_9','MACDs_12_26_9','RSI_14','ATRr_14','STOCHRSIk_14_14_3_3','STOCHRSId_14_14_3_3','ATRr_14','BB_upper','BB_middle','BB_lower','OBV']
stocks_ta = stocks_ta.dropna(subset=columns_to_check)
print(f"Shape after dropping TA NaNs: {stocks_ta.shape}")



Shape after dropping TA NaNs: (104220, 22)


In [6]:
stocks_ta = stocks_ta.reset_index(drop=True)

# 1-day return (t-1 -> t)
stocks_ta['ret_1d'] = stocks_ta.groupby('ticker')['close'].pct_change()
# 1-day rolling return is the same as ret_1d; keep a named column for clarity
stocks_ta['roll_ret_1d'] = stocks_ta['ret_1d']
# 5-day rolling mean of past 1-day returns
stocks_ta['roll_ret_5d'] = (
    stocks_ta.groupby('ticker')['ret_1d']
    .transform(lambda s: s.rolling(5, min_periods=1).mean())
)
# 20-day rolling mean of past 1-day returns
stocks_ta['roll_ret_20d'] = (
    stocks_ta.groupby('ticker')['ret_1d']
    .transform(lambda s: s.rolling(20, min_periods=1).mean())
)
print(f"Shape after return features: {stocks_ta.shape}")



Shape after return features: (104220, 26)


In [7]:
stocks_ta.reset_index(drop=True, inplace=True)
stocks_ta

,date,open,high,low,close,adj close,volume,ticker,EMA_12,EMA_26,...,STOCHRSId_14_14_3_3,ATRr_14,BB_upper,BB_middle,BB_lower,OBV,ret_1d,roll_ret_1d,roll_ret_5d,roll_ret_20d
0,2012-11-14,77.928574,78.207146,76.597145,76.697144,69.613815,119292600.0,AAPL,80.708033,84.949698,...,19.582354,2.377852,94.648550,84.401357,74.154164,-1.014356e+09,NaN,NaN,NaN,NaN
1,2012-11-15,76.790001,77.071426,74.660004,75.088570,68.153778,197477700.0,AAPL,79.843501,84.219244,...,19.993462,2.380310,93.761634,83.514428,73.267223,-1.211834e+09,-0.020973,-0.020973,-0.020973,-0.020973
2,2012-11-16,75.028572,75.714287,72.250000,75.382858,68.420891,316723400.0,AAPL,79.157248,83.564697,...,16.363641,2.459547,92.716200,82.679214,72.642228,-8.951103e+08,0.003919,0.003919,-0.008527,-0.008527
3,2012-11-19,77.244286,81.071426,77.125717,80.818573,73.354591,205829400.0,AAPL,79.412836,83.361280,...,22.576123,2.695187,91.617665,82.201285,72.784906,-6.892809e+08,0.072108,0.072108,0.018351,0.018351
4,2012-11-20,81.701431,81.707146,79.225716,80.129997,72.729614,160688500.0,AAPL,79.523169,83.121926,...,40.436207,2.679612,91.027780,81.851785,72.675790,-8.499694e+08,-0.008520,-0.008520,0.011634,0.011634
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
104215,2017-08-28,76.900002,76.940002,76.260002,76.470001,76.470001,8229700.0,XOM,77.187452,78.267858,...,31.775404,0.786087,81.525829,78.243500,74.961171,-2.688251e+08,-0.003259,-0.003259,0.000243,-0.002261
104216,2017-08-29,76.209999,76.489998,76.080002,76.449997,76.449997,7060400.0,XOM,77.073998,78.133202,...,38.712818,0.759224,81.303475,78.057500,74.811525,-2.758855e+08,-0.000262,-0.000262,-0.000752,-0.002355
104217,2017-08-30,76.239998,76.449997,76.059998,76.099998,76.099998,8218000.0,XOM,76.924151,77.982594,...,46.246701,0.732850,80.964170,77.832500,74.700830,-2.841035e+08,-0.004578,-0.004578,-0.001329,-0.002852
104218,2017-08-31,76.269997,76.489998,76.050003,76.330002,76.330002,15641700.0,XOM,76.832744,77.860180,...,59.187972,0.711932,80.569554,77.624500,74.679446,-2.684618e+08,0.003022,0.003022,0.000007,-0.002633


In [8]:
stocks_ta.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 104220 entries, 0 to 104219
Data columns (total 26 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   date                 104220 non-null  datetime64[ns]
 1   open                 104220 non-null  float64       
 2   high                 104220 non-null  float64       
 3   low                  104220 non-null  float64       
 4   close                104220 non-null  float64       
 5   adj close            104220 non-null  float64       
 6   volume               104220 non-null  float64       
 7   ticker               104220 non-null  object        
 8   EMA_12               104220 non-null  float64       
 9   EMA_26               104220 non-null  float64       
 10  EMA_50               104220 non-null  float64       
 11  MACD_12_26_9         104220 non-null  float64       
 12  MACDh_12_26_9        104220 non-null  float64       
 13  MACDs_12_26_9 

In [9]:
tweets = pd.read_parquet('../data/dataset/stock_tweets_sentiment_emotion_stanceScore_finbert_nomerge.parquet')

print(f"Shape of tweets raw: {tweets.shape}")

tweets.columns = [x.lower() for x in tweets.columns]

# Normalize stance/finbert labels to numeric
if 'stance_label' in tweets.columns:
    stance_label = tweets['stance_label'].astype(str).str.lower()
    tweets['stance_label'] = stance_label.map({'positive': 1, 'neutral': 0, 'negative': -1})

if 'finbert_label' in tweets.columns:
    finbert_label = tweets['finbert_label'].astype(str).str.lower()
    tweets['finbert_label'] = finbert_label.map({'positive': 1, 'neutral': 0, 'negative': -1})

# Ensure raw emotion columns exist before computing pct/unified
emotion_cols = [
    'emotion_anger', 'emotion_disgust', 'emotion_fear',
    'emotion_joy', 'emotion_neutral', 'emotion_sadness', 'emotion_surprize'
]

# Add percentile emotion features if missing
for c in emotion_cols:
    pct_col = c + '_pct'
    if c in tweets.columns and pct_col not in tweets.columns:
        tweets[pct_col] = tweets[c].rank(pct=True)

# Add unified emotion features if missing
if 'positive_emotion' not in tweets.columns and 'emotion_joy' in tweets.columns:
    tweets['positive_emotion'] = tweets['emotion_joy']
if 'negative_emotion' not in tweets.columns and set(['emotion_anger','emotion_disgust','emotion_sadness']).issubset(tweets.columns):
    tweets['negative_emotion'] = tweets[['emotion_anger','emotion_disgust','emotion_sadness']].sum(axis=1)
if 'uncertainty_emotion' not in tweets.columns and set(['emotion_fear','emotion_surprize']).issubset(tweets.columns):
    tweets['uncertainty_emotion'] = tweets[['emotion_fear','emotion_surprize']].sum(axis=1)

if 'positive_emotion_pct' not in tweets.columns and 'emotion_joy_pct' in tweets.columns:
    tweets['positive_emotion_pct'] = tweets['emotion_joy_pct']
if 'negative_emotion_pct' not in tweets.columns and set(['emotion_anger_pct','emotion_disgust_pct','emotion_sadness_pct']).issubset(tweets.columns):
    tweets['negative_emotion_pct'] = tweets[['emotion_anger_pct','emotion_disgust_pct','emotion_sadness_pct']].sum(axis=1)
if 'uncertainty_emotion_pct' not in tweets.columns and set(['emotion_fear_pct','emotion_surprize_pct']).issubset(tweets.columns):
    tweets['uncertainty_emotion_pct'] = tweets[['emotion_fear_pct','emotion_surprize_pct']].sum(axis=1)

# Ensure date type
if 'date' in tweets.columns:
    tweets['date'] = pd.to_datetime(tweets['date'])

# Aggregate to one row per (date, ticker)
agg = {
    'text': lambda x: ' '.join(x)
}

if 'sentiment' in tweets.columns:
    agg['sentiment'] = 'mean'

for col in emotion_cols:
    if col in tweets.columns:
        agg[col] = 'sum'

for col in [
    'emotion_anger_pct', 'emotion_disgust_pct', 'emotion_fear_pct',
    'emotion_joy_pct', 'emotion_neutral_pct', 'emotion_sadness_pct', 'emotion_surprize_pct'
]:
    if col in tweets.columns:
        agg[col] = 'mean'

for col in ['positive_emotion', 'negative_emotion', 'uncertainty_emotion']:
    if col in tweets.columns:
        agg[col] = 'sum'

for col in ['positive_emotion_pct', 'negative_emotion_pct', 'uncertainty_emotion_pct']:
    if col in tweets.columns:
        agg[col] = 'mean'

# stance/finbert labels already numeric; aggregate by mean
for col in ['stance_label', 'finbert_label']:
    if col in tweets.columns:
        agg[col] = 'mean'

for col in ['stance_score', 'finbert_score', 'finbert_up', 'finbert_down', 'finbert_neutral']:
    if col in tweets.columns:
        agg[col] = 'mean'


tweets_merged = tweets.groupby(['date', 'ticker'], as_index=False).agg(agg)

print(f"Shape after tweet aggregation: {tweets_merged.shape}")

price_tweets = pd.merge(
    stocks_ta,
    tweets_merged,
    on=["date", "ticker"],
    how='left'
)

print(f"Shape after merge with TA: {price_tweets.shape}")

# Fill missing NLP features with 0 (no-tweet days)
nlp_cols = [
    'sentiment',
    'emotion_anger', 'emotion_disgust', 'emotion_fear', 'emotion_joy', 'emotion_neutral', 'emotion_sadness', 'emotion_surprize',
    'emotion_anger_pct', 'emotion_disgust_pct', 'emotion_fear_pct', 'emotion_joy_pct', 'emotion_neutral_pct', 'emotion_sadness_pct', 'emotion_surprize_pct',
    'positive_emotion', 'negative_emotion', 'uncertainty_emotion',
    'positive_emotion_pct', 'negative_emotion_pct', 'uncertainty_emotion_pct',
    'stance_label', 'stance_score',
    'finbert_label', 'finbert_score', 'finbert_up', 'finbert_down', 'finbert_neutral'
]
cols_to_fill = [c for c in nlp_cols if c in price_tweets.columns]
price_tweets[cols_to_fill] = price_tweets[cols_to_fill].fillna(0)

price_tweets = pd.merge(price_tweets, companies[['ticker', 'sector', 'company']], on='ticker', how='left')

price_tweets = price_tweets.rename(columns={'close': 'close_price', 'company': 'company_name'})

print(f"Shape after company merge: {price_tweets.shape}")

price_tweets.rename(columns={'close_price': 'close'}, inplace=True)

price_tweets.sort_values(by=['ticker', 'date'], inplace=True)


Shape of tweets raw: (106338, 33)
Shape after tweet aggregation: (25658, 31)
Shape after merge with TA: (104220, 55)
Shape after company merge: (104220, 57)


In [10]:
price_tweets


,date,open,high,low,close,adj close,volume,ticker,EMA_12,EMA_26,...,uncertainty_emotion_pct,stance_label,finbert_label,stance_score,finbert_score,finbert_up,finbert_down,finbert_neutral,sector,company_name
0,2012-11-14,77.928574,78.207146,76.597145,76.697144,69.613815,119292600.0,AAPL,80.708033,84.949698,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Consumer Goods,Apple Inc.
1,2012-11-15,76.790001,77.071426,74.660004,75.088570,68.153778,197477700.0,AAPL,79.843501,84.219244,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Consumer Goods,Apple Inc.
2,2012-11-16,75.028572,75.714287,72.250000,75.382858,68.420891,316723400.0,AAPL,79.157248,83.564697,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Consumer Goods,Apple Inc.
3,2012-11-19,77.244286,81.071426,77.125717,80.818573,73.354591,205829400.0,AAPL,79.412836,83.361280,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Consumer Goods,Apple Inc.
4,2012-11-20,81.701431,81.707146,79.225716,80.129997,72.729614,160688500.0,AAPL,79.523169,83.121926,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Consumer Goods,Apple Inc.
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
104215,2017-08-28,76.900002,76.940002,76.260002,76.470001,76.470001,8229700.0,XOM,77.187452,78.267858,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Basic Matierials,Exxon Mobil Corporation
104216,2017-08-29,76.209999,76.489998,76.080002,76.449997,76.449997,7060400.0,XOM,77.073998,78.133202,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Basic Matierials,Exxon Mobil Corporation
104217,2017-08-30,76.239998,76.449997,76.059998,76.099998,76.099998,8218000.0,XOM,76.924151,77.982594,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Basic Matierials,Exxon Mobil Corporation
104218,2017-08-31,76.269997,76.489998,76.050003,76.330002,76.330002,15641700.0,XOM,76.832744,77.860180,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Basic Matierials,Exxon Mobil Corporation


In [12]:
print((price_tweets['finbert_score'] == 0).sum())

84923


In [11]:
price_tweets.to_parquet('../data/dataset/ta_nlp.parquet', index=False)